# 02 Real Pipeline Readiness Gate

This notebook checks the real experiment configuration end to end: configured datasets load real samples, configured foundation model repositories are reachable, and the model-dataset matrix is explicit before any GPU training run.

It intentionally does not run the old toy `smoke_toy` training loop.


In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT


PosixPath('/Users/similoluwaokunowo/Desktop/Research/geoai-lightning-talk-research')

In [2]:
import pandas as pd

from eval_harness.config import load_yaml, dataset_config_path, model_config_path
from eval_harness.readiness import check_experiment_ready

experiment = load_yaml(PROJECT_ROOT / 'config' / 'experiment.yaml')
experiment


{'run_id': 'geoai_deployment_readiness_v1',
 'seed': 1234,
 'device': 'auto',
 'datasets': ['sen1floods11_ghana', 'ftw_africa'],
 'models': ['prithvi_eo_v2_300m', 'terramind_v1_base', 'terramind_v1_large'],
 'adaptation': {'methods': ['linear_probe', 'lora'],
  'lora': {'rank': 8, 'alpha': 16, 'dropout': 0.05}},
 'label_budgets': [0.05, 0.1, 0.25, 1.0],
 'seeds': [1234, 2026, 777],
 'max_epochs': 5,
 'batch_size': 16,
 'num_workers': 4}

In [3]:
load_weights = os.environ.get('LOAD_MODEL_WEIGHTS') == '1'
results = check_experiment_ready(experiment, load_weights=load_weights)
rows = [result.as_dict() for result in results]
readiness = pd.DataFrame([
    {
        'kind': row['kind'],
        'name': row['name'],
        'ready': row['ready'],
        'error': row['error'],
    }
    for row in rows
])
display(readiness)

if not readiness['ready'].all():
    raise RuntimeError('Readiness gate failed. See error column above.')


,kind,name,ready,error
0,dataset,sen1floods11_ghana,True,None
1,dataset,ftw_africa,True,None
2,model,prithvi_eo_v2_300m,True,None
3,model,terramind_v1_base,True,None
4,model,terramind_v1_large,True,None


In [4]:
dataset_rows = []
for result in results:
    if result.kind != 'dataset':
        continue
    for split, info in result.details['splits'].items():
        dataset_rows.append({
            'dataset': result.name,
            'split': split,
            'num_examples': info['num_examples'],
            'sample_id': info['sample_id'],
            'x_shape': tuple(info['x_shape']),
            'y_shape': tuple(info['y_shape']),
        })
display(pd.DataFrame(dataset_rows))


,dataset,split,num_examples,sample_id,x_shape,y_shape
0,sen1floods11_ghana,train,37,Ghana_103272,"(13, 512, 512)","(512, 512)"
1,sen1floods11_ghana,val,8,Ghana_49890,"(13, 512, 512)","(512, 512)"
2,sen1floods11_ghana,test,8,Ghana_83483,"(13, 512, 512)","(512, 512)"
3,ftw_africa,train,796,south_africa/g1_00008_1,"(8, 512, 512)","(512, 512)"
4,ftw_africa,val,170,south_africa/g3_00006_15,"(8, 512, 512)","(512, 512)"
5,ftw_africa,test,171,kenya/g0_0000008704-0000003584,"(8, 512, 512)","(512, 512)"


In [5]:
model_rows = []
for result in results:
    if result.kind != 'model':
        continue
    details = result.details
    model_rows.append({
        'model': result.name,
        'display_name': details['display_name'],
        'hf_repo': details['hf_repo'],
        'parameter_count_estimate': details['parameter_count_estimate'],
        'checkpoint_size_estimate': details['checkpoint_size_estimate'],
        'weight_files': details['weight_files'],
        'checkpoint_load_ok': details['checkpoint_load_ok'],
    })
display(pd.DataFrame(model_rows))


,model,display_name,hf_repo,parameter_count_estimate,checkpoint_size_estimate,weight_files,checkpoint_load_ok
0,prithvi_eo_v2_300m,Prithvi-EO-2.0 300M,ibm-nasa-geospatial/Prithvi-EO-2.0-300M,300000000,unknown,[Prithvi_EO_V2_300M.pt],None
1,terramind_v1_base,TerraMind-1.0 Base,ibm-esa-geospatial/TerraMind-1.0-base,380000000,1.52GB,[TerraMind_v1_base.pt],None
2,terramind_v1_large,TerraMind-1.0 Large,ibm-esa-geospatial/TerraMind-1.0-large,950000000,3.79GB,[TerraMind_v1_large.pt],None


In [6]:
matrix = []
for dataset_name in experiment['datasets']:
    dataset_cfg = load_yaml(dataset_config_path(dataset_name))
    for model_name in experiment['models']:
        model_cfg = load_yaml(model_config_path(model_name))
        matrix.append({
            'dataset': dataset_name,
            'task_type': dataset_cfg['task_type'],
            'theme': dataset_cfg['climate_theme'],
            'model': model_name,
            'adaptation_methods': ', '.join(experiment['adaptation']['methods']),
            'label_budgets': experiment['label_budgets'],
            'seeds': experiment['seeds'],
            'gpu_machine_type': model_cfg['machine_type'],
        })
display(pd.DataFrame(matrix))


,dataset,task_type,theme,model,adaptation_methods,label_budgets,seeds,gpu_machine_type
0,sen1floods11_ghana,segmentation,Disaster response and humanitarian flood mapping,prithvi_eo_v2_300m,"linear_probe, lora","[0.05, 0.1, 0.25, 1.0]","[1234, 2026, 777]",a2-highgpu-1g
1,sen1floods11_ghana,segmentation,Disaster response and humanitarian flood mapping,terramind_v1_base,"linear_probe, lora","[0.05, 0.1, 0.25, 1.0]","[1234, 2026, 777]",a2-highgpu-1g
2,sen1floods11_ghana,segmentation,Disaster response and humanitarian flood mapping,terramind_v1_large,"linear_probe, lora","[0.05, 0.1, 0.25, 1.0]","[1234, 2026, 777]",a2-ultragpu-1g
3,ftw_africa,segmentation,Food security and agricultural field-boundary ...,prithvi_eo_v2_300m,"linear_probe, lora","[0.05, 0.1, 0.25, 1.0]","[1234, 2026, 777]",a2-highgpu-1g
4,ftw_africa,segmentation,Food security and agricultural field-boundary ...,terramind_v1_base,"linear_probe, lora","[0.05, 0.1, 0.25, 1.0]","[1234, 2026, 777]",a2-highgpu-1g
5,ftw_africa,segmentation,Food security and agricultural field-boundary ...,terramind_v1_large,"linear_probe, lora","[0.05, 0.1, 0.25, 1.0]","[1234, 2026, 777]",a2-ultragpu-1g


## Pass Criteria

- Every dataset in `config/experiment.yaml` is real-loader ready.
- Every foundation model repo in `config/experiment.yaml` is reachable and exposes the expected checkpoint files.
- The model-dataset-adaptation matrix is visible before GPU jobs are launched.
- Optional heavyweight checkpoint loading passes when `LOAD_MODEL_WEIGHTS=1` is set on a suitable machine.
